# Meta Model


## Purged Validation

- **Purpose:** Apply `purged_validation.py`'s `PurgedKFold` directly to the development partition.
- **Settings:** `cv=5` contiguous folds; `pct_embargo=0.01`; `random_state=42`; fixed chronological holdout.
- **Data:** Use development meta features, action labels, sample weights, and event information intervals.
- **Decision:** Purge overlapping intervals, exclude holdout from every fold, and maximize weighted F1 with weighted log loss used for ties.


In [ ]:
from pathlib import Path

import joblib
from IPython import get_ipython

get_ipython().run_line_magic("matplotlib", "inline")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)

PROJECT_ROOT = Path.cwd().resolve().parents[1]

from src.modeling.purged_validation import index_events
from src.modeling.purged_validation import PurgedKFold
from src.modeling.feature_importance import (
    get_estimator_feature_importance,
)
from src.modeling.hyperparameter_tuning import (
    fit_classifier_with_hyperparameter_search,
)
from src.modeling.model_workflow import (
    build_candidate_classifiers,
    build_meta_model_frame,
    candidate_parameter_grids,
    generate_oof_predictions,
    get_meta_feature_columns,
    get_weighted_learning_curve,
    score_binary_predictions,
)

RANDOM_STATE = 42
CV_SPLITS = 5
PCT_EMBARGO = 0.01
TRAIN_SIZES = [0.20, 0.40, 0.60, 0.80, 1.00]
period = "2025-02-01_2025-12-31"
event_path = PROJECT_ROOT / f"data/research_data/events/sp500_model_events_{period}.parquet"
artifact_dir = PROJECT_ROOT / "data/model_artifact"
artifact_dir.mkdir(parents=True, exist_ok=True)

events = index_events(pd.read_parquet(event_path))
split_manifest = pd.read_parquet(artifact_dir / "split_manifest.parquet")
primary_predictions = index_events(pd.read_parquet(artifact_dir / "primary_predictions.parquet"))

development_primary = primary_predictions[
    primary_predictions["partition"].eq("development")
]
manifest_development = pd.MultiIndex.from_frame(split_manifest.loc[
    split_manifest["partition"].eq("development"), ["symbol", "event_start"]
])
if set(development_primary.index) != set(manifest_development):
    raise ValueError("Primary OOF rows must match the fixed development partition")

development_events = events.loc[development_primary.index]
primary_oof_contract = development_primary.rename(columns={
    "primary_side": "prediction",
    "primary_probability": "probability",
})
meta_development = build_meta_model_frame(
    development_events,
    primary_oof_contract[["prediction", "probability", "prediction_source"]],
)
meta_features = get_meta_feature_columns(meta_development)

X_development = meta_development[meta_features]
y_development = meta_development["meta_label"].astype("int8")
w_development = meta_development["sample_weight"].astype("float64")
t1_development = meta_development["event_end"]
CLASS_LABELS = [0, 1]
LOG_LOSS_COLUMN = "log_loss"
SELECTION_COLUMNS = ["f1", "log_loss"]
SELECTION_ASCENDING = [False, True]
LEARNING_SCORING = "f1"
IMPORTANCE_SCORING = "f1"

cv = PurgedKFold(CV_SPLITS, t1=t1_development, pct_embargo=PCT_EMBARGO)
fold_rows = []
for fold, (train, test) in enumerate(cv.split(X_development)):
    fold_rows.append({
        "fold": fold,
        "train_events": len(train),
        "test_events": len(test),
        "test_start": t1_development.index.get_level_values("event_start")[test].min(),
        "test_end": t1_development.iloc[test].max(),
    })

display(pd.DataFrame(fold_rows).set_index("fold"))
display(pd.Series({
    "development_events": len(meta_development),
    "act_labels": int(y_development.sum()),
    "pass_labels": int((1 - y_development).sum()),
    "features": len(meta_features),
}, name="value").to_frame())


## Ensemble Methods

- **Purpose:** Compare the boosting, bagging, and random-forest factories from `ensemble_methods.py`.
- **Settings:** `boosting_stages=100`; `bagging_estimators=120`; `rf_estimators=120`; `random_state=42`; `n_jobs=1`.
- **Data:** Generate comparable purged OOF predictions for all three candidates from the development partition.
- **Decision:** Rank candidates by maximum weighted F1 and use weighted log loss for ties.


In [ ]:
candidates = build_candidate_classifiers(
    random_state=RANDOM_STATE,
    n_jobs=1,
)

candidate_oof = {}
comparison_rows = []
for name, estimator in candidates.items():
    predictions = generate_oof_predictions(
        estimator,
        X_development,
        y_development,
        w_development,
        cv,
        positive_label=1,
    )
    candidate_oof[name] = predictions
    scores = score_binary_predictions(
        y_development,
        predictions["prediction"],
        predictions["probability"],
        w_development,
        class_labels=CLASS_LABELS,
        positive_label=1,
    )
    comparison_rows.append({
        "candidate": name,
        "f1": scores["f1"],
        LOG_LOSS_COLUMN: scores["log_loss"],
    })

comparison = pd.DataFrame(comparison_rows).set_index("candidate")
ranked_comparison = comparison.sort_values(SELECTION_COLUMNS, ascending=SELECTION_ASCENDING)
selected_name = ranked_comparison.index[0]
selected_estimator = candidates[selected_name]
display(comparison.rename(index=lambda name: name.replace("_", " ").title()))
print(f"Selected baseline family: {selected_name}")


## Ensemble Methods Learning Curves

- **Purpose:** Use `get_weighted_learning_curve` to compare how the three baseline candidates learn across purged folds.
- **Settings:** `train_sizes=(0.20, 0.40, 0.60, 0.80, 1.00)`; `cv=5`; error is `1 - weighted F1` with mean and standard error.
- **Data:** Build learning curves from development folds only.
- **Decision:** Interpret the train-validation gap as a diagnostic rather than a model-selection result.


In [ ]:
def plot_learning_curves(estimators, heading):
    results = {}
    model_count = len(estimators)
    grid_size = 1 if model_count == 1 else 2
    figure_size = (7, 5.5) if model_count == 1 else (14, 9)
    fig, axes = plt.subplots(
        grid_size,
        grid_size,
        figsize=figure_size,
        sharex=False,
        sharey=False,
    )
    axes = np.atleast_1d(axes).ravel()
    for axis, (name, estimator) in zip(axes, estimators.items()):
        curve = get_weighted_learning_curve(
            estimator,
            X_development,
            y_development,
            w_development,
            cv,
            train_sizes=TRAIN_SIZES,
            class_labels=CLASS_LABELS,
            positive_label=1,
            scoring=LEARNING_SCORING,
            random_state=RANDOM_STATE,
        )
        results[name] = curve
        x = curve["train_size"].to_numpy()
        for split, color in [("train", "tab:red"), ("validation", "tab:blue")]:
            mean = curve[f"{split}_error_mean"].to_numpy()
            error = curve[f"{split}_error_std"].fillna(0.0).to_numpy()
            axis.plot(x, mean, marker="o", color=color, label=split)
            axis.fill_between(x, mean - error, mean + error, color=color, alpha=0.15)
        axis.set_title(name.replace("_", " ").title())
        axis.set_xlabel("Training set size")
        axis.set_ylabel("Weighted error")
        axis.grid(alpha=0.25)
        axis.legend()
    for unused_axis in axes[model_count:]:
        unused_axis.remove()
    fig.suptitle(heading, fontsize=14)
    fig.tight_layout()
    plt.show()
    return results


ensemble_learning_curves = plot_learning_curves(
    candidates,
    "Ensemble Methods Learning Curves",
)


## Ensemble Methods Feature Importance

- **Purpose:** Apply estimator-aware MDI, MDA, and SFI to the selected baseline family.
- **Settings:** `cv=5`; `pct_embargo=0.01`; `random_state=42`; `top_k=15`; MDA and SFI use weighted F1.
- **Data:** Calculate and preserve importance results for all 56 meta features.
- **Decision:** Treat importance as a diagnostic rather than causal evidence or a new selection step.


In [ ]:
def compute_stage_importance(estimator):
    results = {}
    scores = {}
    for method in ["MDI", "MDA", "SFI"]:
        importance, oos = get_estimator_feature_importance(
            estimator,
            X_development,
            y_development,
            w_development,
            t1_development,
            method=method,
            scoring=IMPORTANCE_SCORING,
            cv=CV_SPLITS,
            pct_embargo=PCT_EMBARGO,
            random_state=RANDOM_STATE,
        )
        results[method] = importance
        scores[method] = oos
    return results, pd.Series(scores, name="oos_score")


def plot_feature_importance(results, heading):
    fig, axes = plt.subplots(1, 3, figsize=(19, 8))
    for axis, method in zip(axes, ["MDI", "MDA", "SFI"]):
        importance = results[method].nlargest(15, "mean").sort_values("mean")
        axis.barh(
            importance.index,
            importance["mean"],
            xerr=importance["std"].fillna(0.0),
            alpha=0.75,
            color="tab:blue",
        )
        axis.axvline(0.0, color="black", linewidth=0.8)
        axis.set_title(method)
        axis.set_xlabel("Mean importance or score")
        axis.grid(axis="x", alpha=0.25)
    fig.suptitle(heading, fontsize=14)
    fig.tight_layout()
    plt.show()


ensemble_importance_results, ensemble_importance_scores = compute_stage_importance(
    selected_estimator
)
plot_feature_importance(
    ensemble_importance_results,
    f"Ensemble Methods Feature Importance: {selected_name}",
)
display(ensemble_importance_scores.to_frame())


## Ensemble Methods Model Evaluation

- **Purpose:** Compare baseline weighted accuracy, precision, recall, F1, log loss, confusion matrices, PR curves, and ROC curves.
- **Settings:** Metric order is accuracy, precision, recall, F1, then weighted log loss; `positive_label=1`; weighted confusion matrices, PR, ROC, AP, and ROC AUC.
- **Data:** Evaluate the development OOF predictions for all three candidates.
- **Decision:** Do not use holdout curves to adjust thresholds.


In [ ]:
def plot_model_evaluation(predictions_by_model, observed, weights, heading):
    metric_rows = []
    for name, predictions in predictions_by_model.items():
        scores = score_binary_predictions(
            observed,
            predictions["prediction"],
            predictions["probability"],
            weights,
            class_labels=CLASS_LABELS,
            positive_label=1,
        )
        metric_rows.append({
            "model": name,
            "accuracy": scores["accuracy"],
            "precision": scores["precision"],
            "recall": scores["recall"],
            "f1": scores["f1"],
            LOG_LOSS_COLUMN: scores["log_loss"],
        })
    display(pd.DataFrame(metric_rows).set_index("model").rename(index=lambda name: name.replace("_", " ").title()))

    model_count = len(predictions_by_model)
    grid_size = 1 if model_count == 1 else 2
    figure_size = (7, 5.5) if model_count == 1 else (10, 9)
    fig, axes = plt.subplots(grid_size, grid_size, figsize=figure_size)
    axes = np.atleast_1d(axes).ravel()
    for axis, (name, predictions) in zip(axes, predictions_by_model.items()):
        matrix = confusion_matrix(
            observed,
            predictions["prediction"],
            labels=CLASS_LABELS,
            sample_weight=weights,
            normalize="true",
        )
        ConfusionMatrixDisplay(matrix, display_labels=CLASS_LABELS).plot(
            ax=axis,
            colorbar=False,
            values_format=".2f",
        )
        axis.set_title(name.replace("_", " ").title())
    for unused_axis in axes[model_count:]:
        unused_axis.remove()
    fig.suptitle(f"{heading}: Weighted Confusion Matrices", fontsize=14)
    fig.tight_layout()
    plt.show()

    fig, (pr_axis, roc_axis) = plt.subplots(1, 2, figsize=(14, 5))
    for name, predictions in predictions_by_model.items():
        probability = predictions["probability"]
        precision, recall, _ = precision_recall_curve(
            observed,
            probability,
            pos_label=1,
            sample_weight=weights,
        )
        average_precision = average_precision_score(
            observed,
            probability,
            pos_label=1,
            sample_weight=weights,
        )
        pr_axis.plot(recall, precision, label=f"{name.replace('_', ' ').title()} (AP={average_precision:.3f})")

        false_positive_rate, true_positive_rate, _ = roc_curve(
            observed,
            probability,
            pos_label=1,
            sample_weight=weights,
        )
        auc = roc_auc_score(observed, probability, sample_weight=weights)
        roc_axis.plot(
            false_positive_rate,
            true_positive_rate,
            label=f"{name.replace('_', ' ').title()} (AUC={auc:.3f})",
        )

    pr_axis.set(xlabel="Recall", ylabel="Precision", title="Precision-Recall")
    pr_axis.grid(alpha=0.25)
    pr_axis.legend(fontsize=8)
    roc_axis.plot([0, 1], [0, 1], "k--", label="Random")
    roc_axis.set(xlabel="False positive rate", ylabel="True positive rate", title="ROC")
    roc_axis.grid(alpha=0.25)
    roc_axis.legend(fontsize=8)
    fig.suptitle(heading, fontsize=14)
    fig.tight_layout()
    plt.show()


plot_model_evaluation(
    candidate_oof,
    y_development,
    w_development,
    "Ensemble Methods Model Evaluation",
)


## Hyperparameter Tuning

- **Purpose:** Use `fit_classifier_with_hyperparameter_search` to tune only the family selected by the baseline comparison.
- **Settings:** `cv=5`; `pct_embargo=0.01`; selected-family grids cover bagging `max_samples`, RF `max_features`, or boosting `learning_rate` at three fixed values.
- **Data:** Apply sample weights within purged development folds for the selected family.
- **Decision:** Maximize weighted F1, use weighted log loss for ties, and accept no holdout feedback.


In [ ]:
def to_grid_search_space(configurations):
    grid = {}
    for configuration in configurations:
        for parameter, value in configuration.items():
            grid.setdefault(parameter, []).append(value)
    return grid


parameter_grids = candidate_parameter_grids()
parameter_grid = to_grid_search_space(parameter_grids[selected_name])
fitted = fit_classifier_with_hyperparameter_search(
    X_development,
    y_development,
    t1_development,
    selected_estimator,
    parameter_grid,
    cv=CV_SPLITS,
    n_jobs=1,
    pct_embargo=PCT_EMBARGO,
    sample_weight=w_development.to_numpy(),
)
final_name = selected_name
final_configuration = {
    parameter: fitted.get_params()[parameter]
    for parameter in parameter_grid
}
final_estimator = clone(fitted)
final_oof = generate_oof_predictions(
    final_estimator,
    X_development,
    y_development,
    w_development,
    cv,
    positive_label=1,
)
scores = score_binary_predictions(
    y_development,
    final_oof["prediction"],
    final_oof["probability"],
    w_development,
    class_labels=CLASS_LABELS,
    positive_label=1,
)
tuning_columns = [
    "candidate",
    "configuration",
    "model__max_samples",
    "model__max_features",
    "model__learning_rate",
    "f1",
    LOG_LOSS_COLUMN,
]
tuning = pd.DataFrame([{
    "candidate": final_name,
    "configuration": repr(final_configuration),
    **final_configuration,
    "f1": scores["f1"],
    LOG_LOSS_COLUMN: scores["log_loss"],
}]).reindex(columns=tuning_columns)
tuned_estimators = {f"{final_name}_tuned": final_estimator}
display(tuning)
print(f"Selected tuned family: {final_name}")


## Hyperparameter Tuning Learning Curves

- **Purpose:** Diagnose the selected tuned estimator with the same weighted learning-curve helper.
- **Settings:** `train_sizes=(0.20, 0.40, 0.60, 0.80, 1.00)`; `cv=5`; `pct_embargo=0.01`; error is `1 - weighted F1`.
- **Data:** Build the tuned estimator's learning curve from development folds only.
- **Decision:** Do not revise the model or grid from holdout learning behavior.


In [ ]:
tuned_learning_curves = plot_learning_curves(
    tuned_estimators,
    "Hyperparameter Tuning Learning Curves",
)


## Hyperparameter Tuning Feature Importance

- **Purpose:** Apply the same estimator-aware MDI, MDA, and SFI methods to the final tuned family.
- **Settings:** `cv=5`; `pct_embargo=0.01`; `random_state=42`; `top_k=15`; MDA and SFI use weighted F1.
- **Data:** Calculate and preserve tuned-model importance results for all 56 meta features.
- **Decision:** Treat changes from baseline as diagnostics rather than feature selection.


In [ ]:
tuned_importance_results, tuned_importance_scores = compute_stage_importance(
    final_estimator
)
plot_feature_importance(
    tuned_importance_results,
    f"Hyperparameter Tuning Feature Importance: {final_name}",
)
importance = pd.DataFrame({
    "mdi": tuned_importance_results["MDI"]["mean"],
    "mda": tuned_importance_results["MDA"]["mean"],
    "sfi_neg_log_loss" if IMPORTANCE_SCORING == "neg_log_loss" else "sfi_f1": (
        tuned_importance_results["SFI"]["mean"]
    ),
}).sort_values("mda", ascending=False)
display(importance.head(15))
display(tuned_importance_scores.to_frame())


## Hyperparameter Tuning Model Evaluation

- **Purpose:** Compare the selected family's baseline and tuned accuracy, precision, recall, F1, log loss, and diagnostic curves before final evaluation.
- **Settings:** Metric order is accuracy, precision, recall, F1, then weighted log loss; confusion matrix, PR, and ROC remain weighted; `holdout_evaluations=1`.
- **Data:** Use development OOF results and one final holdout evaluation, then persist the model and prediction provenance.
- **Decision:** Fix the configuration from development OOF and evaluate holdout exactly once.


In [ ]:
tuning_evaluation_oof = {
    f"{selected_name}_baseline": candidate_oof[selected_name],
    f"{final_name}_tuned": final_oof,
}
plot_model_evaluation(
    tuning_evaluation_oof,
    y_development,
    w_development,
    "Hyperparameter Tuning Model Evaluation",
)

final_meta = clone(final_estimator).fit(
    X_development,
    y_development,
    sample_weight=w_development.to_numpy(),
)

holdout_primary = primary_predictions[
    primary_predictions["partition"].eq("holdout")
]
holdout_events = events.loc[holdout_primary.index].copy()
holdout_events["primary_side"] = holdout_primary["primary_side"].astype("int8")
holdout_events["primary_probability"] = holdout_primary["primary_probability"]
holdout_events["primary_confidence"] = holdout_primary["primary_confidence"]
holdout_events["meta_label"] = (
    holdout_events["primary_side"] * holdout_events["raw_return"] > 0
).astype("int8")

holdout_probability = final_meta.predict_proba(
    holdout_events[meta_features]
)[:, list(final_meta.classes_).index(1)]
holdout_action = final_meta.predict(holdout_events[meta_features]).astype("int8")

meta_oof_output = meta_development[[
    "event_end", "raw_return", "direction_label", "sample_weight",
    "primary_side", "primary_probability", "primary_confidence", "meta_label",
]].copy()
meta_oof_output["partition"] = "development"
meta_oof_output["meta_action"] = final_oof["prediction"].astype("int8")
meta_oof_output["meta_probability"] = final_oof["probability"]
meta_oof_output["prediction_source"] = final_oof["prediction_source"]
meta_oof_output["cv_fold"] = final_oof["fold"]

meta_holdout_output = holdout_events[[
    "event_end", "raw_return", "direction_label", "sample_weight",
    "primary_side", "primary_probability", "primary_confidence", "meta_label",
]].copy()
meta_holdout_output["partition"] = "holdout"
meta_holdout_output["meta_action"] = holdout_action
meta_holdout_output["meta_probability"] = holdout_probability
meta_holdout_output["prediction_source"] = "holdout"
meta_holdout_output["cv_fold"] = pd.NA

holdout_prediction_frame = pd.DataFrame({
    "prediction": holdout_action,
    "probability": holdout_probability,
}, index=holdout_events.index)
holdout_scores = score_binary_predictions(
    holdout_events["meta_label"],
    holdout_prediction_frame["prediction"],
    holdout_prediction_frame["probability"],
    holdout_events["sample_weight"],
    class_labels=CLASS_LABELS,
    positive_label=1,
)
holdout_metrics = pd.Series({
    "accuracy": holdout_scores["accuracy"],
    "precision": holdout_scores["precision"],
    "recall": holdout_scores["recall"],
    "f1": holdout_scores["f1"],
    "log_loss": holdout_scores["log_loss"],
}, name="holdout")
display(holdout_metrics.to_frame())
plot_model_evaluation(
    {"final_holdout": holdout_prediction_frame},
    holdout_events["meta_label"],
    holdout_events["sample_weight"],
    "Final One-Time Holdout Evaluation",
)

meta_predictions = pd.concat([
    meta_oof_output,
    meta_holdout_output,
]).sort_index(level=["event_start", "symbol"])
meta_predictions.reset_index().to_parquet(artifact_dir / "meta_predictions.parquet", index=False)
joblib.dump({
    "estimator": final_meta,
    "feature_columns": meta_features,
    "selected_candidate": final_name,
    "best_configuration": final_configuration,
    "random_state": RANDOM_STATE,
}, artifact_dir / "meta_model.joblib")

print(artifact_dir / "meta_model.joblib")
